# Brazil Mid-Market Deals Radar — Phase 3

**A second public source, and a real cross.** Phase 2 read credit stress from the
BCB's SCR.data (non-performing loans by sector). Phase 3 brings an *independent*
public credit lens — **corporate leverage of B3-listed companies**, from the CVM's
open financial statements (DFP) — and cracks it open four ways:

1. **Cross** — do the two lenses agree, sector by sector? (rank correlation + map)
2. **Composite credit score** — one ranked read from both lenses + trend
3. **Divergence** — where they disagree, and what that says
4. **Company drill-down** — the listed names behind each sector's leverage
5. **Leverage trend** — two DFP vintages, crossed with the NPL trend

Real on both credit axes (SCR and CVM are public). Reuses the Phase 2 discipline:
cache-first download, SHA-256, sanity gates that refuse to write on an implausible
result, and committed derived CSVs so a fresh clone runs offline.

> Requirements: `pandas`, `matplotlib`, `requests`, `scipy`. The deal side stays
> synthetic and labeled; nothing here collects deals.

## 0 · Configuration

In [ ]:
import hashlib, time, unicodedata, zipfile
from datetime import date
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

REFRESH_LEVERAGE = False        # True -> download from the CVM and rewrite the CSVs
CVM_YEAR         = 2025         # most recent annual DFP (fiscal year)
BUILD_TREND      = True         # also pull the prior year to measure a leverage trend
CVM_YEAR_PREV    = CVM_YEAR - 1

# Composite-score weights (editable). Higher score = more credit stress.
SCORE_W = {"npl_level": 0.35, "npl_trend": 0.25, "leverage": 0.25, "lev_trend": 0.15}
LEV_METRIC = "nd_equity"        # leverage metric used in the score / divergence / map

CVM_DFP_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{year}.zip"
CVM_CAD_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"
LEV_FILE    = "leverage_indicators.csv"
COMP_FILE   = "companies_leverage.csv"
STRESS_FILE = "stress_indicators.csv"
CACHE       = Path(".cvm_cache")
ENC         = "latin-1"
BAR         = "#3b5b7d"
POS, NEG    = "#b23a3a", "#3b7d5b"

CVM_SOURCE = "CVM Dados Abertos — DFP (consolidated), corporate leverage by sector"
CVM_DOC    = "https://dados.cvm.gov.br/dataset/cia_aberta-doc-dfp"
ACC = {"assets": "1", "cash": "1.01.01", "fin_app": "1.01.02",
       "debt_curr": "2.01.04", "debt_ncurr": "2.02.01", "equity": "2.03", "ebit": "3.05"}

def _norm(s):
    n = unicodedata.normalize("NFKD", str(s))
    return "".join(c for c in n if not unicodedata.combining(c)).strip().lower()

CVM_SECTOR_RULES = [
    ("Serviços Financeiros", ["banco", "segur", "financ", "previd", "intermedi", "credito"]),
    ("Energia",              ["petroleo", "gas", "energia", "eletric", "combustivel"]),
    ("Agronegócio",          ["agric", "acucar", "cana", "agro"]),
    ("Alimentos & Bebidas",  ["aliment", "bebida", "carne", "frigor", "laticin"]),
    ("Saúde",                ["saude", "hospital", "medic", "farma", "diagnost"]),
    ("Transporte & Logística",["transporte", "logist", "rodovi", "portuar", "aere", "ferrov"]),
    ("Tecnologia",           ["software", "tecnolog", "informat", "comunica", "telecom"]),
    ("Educação",             ["educa", "ensino"]),
    ("Construção",           ["constru", "imobili", "incorpor", "cimento"]),
    ("Varejo",               ["comercio", "varejo", "atacado"]),
    ("Indústria",            ["siderur", "metalur", "quimic", "papel", "celulos", "textil",
                              "vestuario", "maquin", "autom", "material", "embalag", "petroquim",
                              "borrach", "eletroeletr", "bens indust", "madeira", "minera"]),
    ("Serviços",             ["servic", "holding", "locac", "aluguel", "diversos"]),
]
def _map_cvm_sector(setor_ativ):
    n = _norm(setor_ativ)
    if not n:
        return None
    for sector, keys in CVM_SECTOR_RULES:
        if any(k in n for k in keys):
            return sector
    return None

## 1 · Download the CVM sources (cache-first)

Streams the annual DFP archive(s) and the cadastral file to `.cvm_cache/`, with
retries, a completeness check and a SHA-256 stamp. Nothing downloads if a valid cache
is present. `REFRESH_LEVERAGE = True` forces a rebuild.

In [ ]:
def _download(url, dest, tries=4):
    CACHE.mkdir(exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not (dest.suffix == ".zip" and not zipfile.is_zipfile(dest)):
        print(f"  cached {dest} ({dest.stat().st_size/1e6:.1f} MB)")
        return dest
    for attempt in range(1, tries + 1):
        try:
            print(f"  downloading {url} (attempt {attempt}/{tries})")
            with requests.get(url, stream=True, timeout=(30, 300)) as r:
                if r.status_code != 200:
                    raise RuntimeError(f"CVM returned HTTP {r.status_code}")
                expected = int(r.headers.get("Content-Length", 0))
                tmp = dest.with_suffix(dest.suffix + ".part"); got = 0
                with open(tmp, "wb") as fh:
                    for block in r.iter_content(chunk_size=1 << 20):
                        fh.write(block); got += len(block)
            if expected and got < expected:
                raise IOError(f"truncated: {got} of {expected} bytes")
            if dest.suffix == ".zip" and not zipfile.is_zipfile(tmp):
                raise IOError("downloaded file is not a valid ZIP")
            tmp.replace(dest)
            print(f"    ok — {dest.stat().st_size/1e6:.1f} MB")
            return dest
        except Exception as e:
            print(f"    failed: {type(e).__name__}: {e}")
            if attempt == tries:
                raise RuntimeError(f"could not download {url} after {tries} attempts. Nothing written.")
            time.sleep(3 * attempt)

def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]

## 2 · Parse statements → company-level leverage

For each reporting company: consolidated Balance Sheet (assets, cash, debt, equity),
Income Statement (EBIT) and indirect Cash Flow (D&A, to approximate EBITDA), scaled by
`ESCALA_MOEDA`, latest version of the fiscal year (`ORDEM_EXERC = ÚLTIMO`).

In [ ]:
def _read_stmt(zf, code, year):
    hits = [n for n in zf.namelist() if f"_{code}_con_{year}" in n and n.lower().endswith(".csv")]
    if not hits:
        raise RuntimeError(f"no {code}_con file for {year} in the DFP archive")
    df = pd.read_csv(zf.open(hits[0]), sep=";", encoding=ENC, dtype=str, low_memory=False)
    df = df[df["ORDEM_EXERC"].map(_norm) == "ultimo"].copy()
    scale = df["ESCALA_MOEDA"].map(lambda x: 1000.0 if _norm(x) == "mil" else 1.0)
    df["val"] = pd.to_numeric(df["VL_CONTA"].str.replace(",", ".", regex=False), errors="coerce") * scale
    df["VERSAO"] = pd.to_numeric(df["VERSAO"], errors="coerce").fillna(1)
    return df

def _acc(df, code):
    sub = df[df["CD_CONTA"] == code].sort_values("VERSAO")
    return sub.groupby("CNPJ_CIA")["val"].last()

def _da(dfc):
    n = dfc["DS_CONTA"].map(_norm)
    m = dfc[n.str.contains("deprecia") | n.str.contains("amortiz")]
    return m.groupby("CNPJ_CIA")["val"].sum().abs()

def company_frame(dfp_zip, year):
    with zipfile.ZipFile(dfp_zip) as zf:
        bpa = _read_stmt(zf, "BPA", year); bpp = _read_stmt(zf, "BPP", year)
        dre = _read_stmt(zf, "DRE", year); dfc = _read_stmt(zf, "DFC_MI", year)
    name = bpa.sort_values("VERSAO").groupby("CNPJ_CIA")["DENOM_CIA"].last()
    comp = pd.DataFrame({
        "name":   name,
        "assets": _acc(bpa, ACC["assets"]),
        "cash":   _acc(bpa, ACC["cash"]).add(_acc(bpa, ACC["fin_app"]), fill_value=0),
        "debt":   _acc(bpp, ACC["debt_curr"]).add(_acc(bpp, ACC["debt_ncurr"]), fill_value=0),
        "equity": _acc(bpp, ACC["equity"]),
        "ebit":   _acc(dre, ACC["ebit"]),
        "da":     _da(dfc),
    })
    return comp

def add_sector(comp, cad):
    sector_by_cnpj = cad.drop_duplicates("CNPJ_CIA", keep="last").set_index("CNPJ_CIA")["SETOR_ATIV"]
    comp = comp.copy()
    comp["setor_ativ"] = comp.index.map(sector_by_cnpj)
    comp["sector"] = comp["setor_ativ"].map(_map_cvm_sector)
    return comp

def company_ratios(comp):
    """Per-company leverage ratios, with guards for non-positive denominators."""
    c = comp.copy()
    c["net_debt"] = c["debt"].fillna(0) - c["cash"].fillna(0)
    c["ebitda"]   = c["ebit"].fillna(0) + c["da"].fillna(0)
    c["gd_assets"] = np.where(c["assets"] > 0, c["debt"] / c["assets"], np.nan)
    c["nd_equity"] = np.where(c["equity"] > 0, c["net_debt"] / c["equity"], np.nan)
    c["nd_ebitda"] = np.where(c["ebitda"] > 0, c["net_debt"] / c["ebitda"], np.nan)
    return c

## 3 · Aggregate by sector and write the CSVs

Three leverage metrics, each **book-weighted** (sum the components across the
sector's companies, then divide — a size-weighted ratio, mirroring Phase 2's
book-weighted NPL). Sanity gates cross-check the aggregate and refuse to write on an
implausible result. Two derived files are written and committed: the sector aggregate
(`leverage_indicators.csv`) and the per-company detail for drill-down
(`companies_leverage.csv`).

In [ ]:
def sector_aggregate(comp, cad, label=""):
    comp = add_sector(comp, cad)
    mapped = comp.dropna(subset=["sector"])
    coverage = 100 * len(mapped) / max(len(comp), 1)
    print(f"  {label}{len(comp)} companies | {len(mapped)} mapped ({coverage:.0f}%)")
    g = (mapped.groupby("sector")
         .agg(n=("assets", "size"), assets=("assets", "sum"), cash=("cash", "sum"),
              debt=("debt", "sum"), equity=("equity", "sum"), ebit=("ebit", "sum"), da=("da", "sum")))
    g["net_debt"] = g["debt"] - g["cash"]
    g["ebitda"]   = g["ebit"] + g["da"]
    g["nd_equity"] = (g["net_debt"] / g["equity"]).round(3)
    g["gd_assets"] = (g["debt"] / g["assets"]).round(3)
    g["nd_ebitda"] = (g["net_debt"] / g["ebitda"]).round(3)
    overall = g["debt"].sum() / g["assets"].sum() if g["assets"].sum() else 0
    print(f"  sectors: {len(g)} | aggregate gross-debt/assets: {overall:.2f}")
    if len(g) < 6:
        raise RuntimeError(f"only {len(g)} sectors mapped — SETOR_ATIV mapping looks broken")
    if not 0.05 <= overall <= 0.60:
        raise RuntimeError(f"aggregate gross-debt/assets {overall:.2f} outside [0.05, 0.60] — check codes/scaling")
    return g, mapped

if REFRESH_LEVERAGE or not Path(LEV_FILE).exists():
    print(f"Rebuilding leverage from CVM DFP {CVM_YEAR}\n")
    dfp = _download(CVM_DFP_URL.format(year=CVM_YEAR), CACHE / f"dfp_cia_aberta_{CVM_YEAR}.zip")
    cadf = _download(CVM_CAD_URL, CACHE / "cad_cia_aberta.csv")
    cad = pd.read_csv(cadf, sep=";", encoding=ENC, dtype=str)
    comp = company_frame(dfp, CVM_YEAR)
    g, mapped = sector_aggregate(comp, cad, label=f"{CVM_YEAR}: ")

    lev_trend = None
    if BUILD_TREND:
        print(f"\nLeverage trend: also pulling DFP {CVM_YEAR_PREV}")
        dfp_prev = _download(CVM_DFP_URL.format(year=CVM_YEAR_PREV), CACHE / f"dfp_cia_aberta_{CVM_YEAR_PREV}.zip")
        comp_prev = company_frame(dfp_prev, CVM_YEAR_PREV)
        g_prev, _ = sector_aggregate(comp_prev, cad, label=f"{CVM_YEAR_PREV}: ")
        lev_trend = (g[LEV_METRIC] - g_prev[LEV_METRIC].reindex(g.index)).round(3)

    out = g[["n", "nd_equity", "gd_assets", "nd_ebitda"]].copy()
    if lev_trend is not None:
        out["lev_trend"] = lev_trend
    out = out.reset_index()
    out["source"] = CVM_SOURCE
    out["reference_period"] = f"DFP {CVM_YEAR}" + (f" vs {CVM_YEAR_PREV}" if lev_trend is not None else "")
    out["source_url"] = CVM_DOC
    out["source_sha256"] = f"{_sha(dfp)} / {_sha(cadf)}"
    out["retrieved_at"] = date.today().isoformat()
    out.to_csv(LEV_FILE, index=False)

    detail = company_ratios(mapped)[["name", "sector", "assets", "debt", "equity",
                                     "gd_assets", "nd_equity", "nd_ebitda"]].round(3)
    detail.to_csv(COMP_FILE, index=False)
    print(f"\nWrote {LEV_FILE} and {COMP_FILE} from real CVM data.")
    leverage = out
else:
    leverage = pd.read_csv(LEV_FILE)
    print(f"Loaded {LEV_FILE} from disk — no download. Set REFRESH_LEVERAGE = True to rebuild.")

print()
cols = ["sector", "n", "nd_equity", "gd_assets", "nd_ebitda"] + (["lev_trend"] if "lev_trend" in leverage else [])
print(leverage[cols].to_string(index=False))

## 4 · The cross — do the two lenses agree?

Merge the SCR stress (Phase 2) with the CVM leverage on `sector`, and test whether
loan-book stress and listed-name leverage move together (Spearman rank correlation).

In [ ]:
from scipy.stats import spearmanr

stress = pd.read_csv(STRESS_FILE) if Path(STRESS_FILE).exists() else None
if stress is None:
    print("stress_indicators.csv not found — run the Phase 2 notebook first.")
    panel = None
else:
    panel = stress[["sector", "npl_level_pct", "npl_trend_12m_pp"]].merge(leverage, on="sector", how="inner")
    print(f"Joined {len(panel)} sectors. "
          f"SCR-only: {sorted(set(stress['sector']) - set(panel['sector'])) or '—'} | "
          f"CVM-only: {sorted(set(leverage['sector']) - set(panel['sector'])) or '—'}\n")
    print("Do the two lenses agree? (Spearman rank correlation)")
    for col, lab in [("nd_equity", "Net debt / Equity"), ("gd_assets", "Gross debt / Assets"),
                     ("nd_ebitda", "Net debt / EBITDA")]:
        rho, p = spearmanr(panel["npl_level_pct"], panel[col])
        print(f"  NPL level vs {lab:<20}: rho = {rho:+.2f}  (p = {p:.2f})")

## 5 · Composite credit score

One ranked read from both lenses. Each signal is z-scored across the joined sectors
(so it is unitless and comparable), then combined with editable weights: NPL level,
NPL 12-month trend, leverage, and — where available — the leverage trend. Higher score
= more credit stress. This is the **origination-priority** view: where a credit lens
would send diligence first.

In [ ]:
def _z(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd and sd > 0 else s * 0.0

if panel is not None and len(panel) >= 4:
    sc = panel.copy()
    sc["z_npl_level"] = _z(sc["npl_level_pct"])
    sc["z_npl_trend"] = _z(sc["npl_trend_12m_pp"])
    sc["z_leverage"]  = _z(sc[LEV_METRIC])
    parts = SCORE_W["npl_level"] * sc["z_npl_level"] + SCORE_W["npl_trend"] * sc["z_npl_trend"] \
            + SCORE_W["leverage"] * sc["z_leverage"]
    if "lev_trend" in sc:
        sc["z_lev_trend"] = _z(sc["lev_trend"])
        parts = parts + SCORE_W["lev_trend"] * sc["z_lev_trend"]
    sc["credit_score"] = parts.round(2)
    sc = sc.sort_values("credit_score", ascending=False).reset_index(drop=True)

    print("Composite credit-stress score (higher = more stressed):\n")
    show = ["sector", "credit_score", "npl_level_pct", "npl_trend_12m_pp", LEV_METRIC] + (["lev_trend"] if "lev_trend" in sc else [])
    print(sc[show].round(2).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8.5, max(3, 0.5 * len(sc) + 1)))
    colors = [POS if v >= 0 else NEG for v in sc["credit_score"][::-1]]
    ax.barh(sc["sector"][::-1], sc["credit_score"][::-1], color=colors)
    ax.axvline(0, color="0.6", lw=1)
    ax.set_xlabel("Composite credit-stress score (z-weighted)")
    ax.set_title("Sector credit-stress ranking — both lenses combined\n"
                 f"weights: NPL level {SCORE_W['npl_level']}, trend {SCORE_W['npl_trend']}, "
                 f"leverage {SCORE_W['leverage']}, lev-trend {SCORE_W['lev_trend']}", loc="left", fontsize=10)
    fig.tight_layout(); plt.show()

## 6 · Divergence — where the lenses disagree

With the rank correlation near zero, the *disagreement* is the story. For each sector
we take z(loan-book stress) − z(listed leverage): strongly **positive** means the
loan book is stressed while the listed names are not (the stress likely sits in
smaller, unlisted borrowers the CVM sample never sees); strongly **negative** means
listed leverage is high while the loan book looks calm.

In [ ]:
if panel is not None and len(panel) >= 4:
    dv = panel.copy()
    dv["divergence"] = (_z(dv["npl_level_pct"]) - _z(dv[LEV_METRIC])).round(2)
    dv = dv.sort_values("divergence", ascending=False).reset_index(drop=True)
    print("Divergence = z(SCR NPL) − z(CVM leverage). "
          "+ = loan-book stress the listed names don't show; − = listed leverage the loan book doesn't.\n")
    print(dv[["sector", "divergence", "npl_level_pct", LEV_METRIC]].round(2).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8.5, max(3, 0.5 * len(dv) + 1)))
    colors = [POS if v >= 0 else NEG for v in dv["divergence"][::-1]]
    ax.barh(dv["sector"][::-1], dv["divergence"][::-1], color=colors)
    ax.axvline(0, color="0.6", lw=1)
    ax.set_xlabel("← listed leverage leads      |      loan-book stress leads →")
    ax.set_title("Where the two credit lenses disagree, by sector", loc="left", fontsize=11)
    fig.tight_layout(); plt.show()

## 7 · The map — corroboration scatter

In [ ]:
if panel is not None and len(panel) >= 4:
    x, y = panel["npl_level_pct"], panel[LEV_METRIC]
    fig, ax = plt.subplots(figsize=(9.4, 6.8))
    ax.axhline(y.mean(), color="0.78", lw=1); ax.axvline(x.mean(), color="0.78", lw=1)
    ax.axhspan(y.mean(), y.max() + abs(y.max()) * 0.3 + 0.2, xmin=0.5, color=POS, alpha=0.07, zorder=0)
    ax.scatter(x, y, s=140, color=BAR, alpha=0.85, edgecolor="white", linewidth=1.2, zorder=3)
    placed = []
    for _, r in panel.iterrows():
        px, py = r["npl_level_pct"], r[LEV_METRIC]
        crowded = any(abs(px - qx) < 0.25 and abs(py - qy) < 0.25 for qx, qy in placed)
        ax.annotate(r["sector"], (px, py), xytext=(0, -18 if crowded else 11),
                    textcoords="offset points", ha="center", va="top" if crowded else "bottom", fontsize=9)
        placed.append((px, py))
    ax.set_xlabel("SCR loan-book stress — NPL level (%)  ·  whole credit universe")
    ax.set_ylabel(f"CVM leverage — {LEV_METRIC}  ·  B3-listed names")
    ax.set_title("Two credit lenses by sector\n"
                 "top-right = both flag stress (corroborated); off-diagonal = the lenses disagree", fontsize=11)
    ax.margins(0.16); fig.tight_layout(); plt.show()

## 8 · Company drill-down — the names behind the leverage

The sector aggregates are book-weighted, so a few large names can drive them. Here we
open the box: the most-levered listed companies in each sector (by net debt / equity,
among those with positive equity). Reads the committed `companies_leverage.csv`, so it
runs offline. `drill("Energia")` inspects any single sector.

In [ ]:
detail = pd.read_csv(COMP_FILE) if Path(COMP_FILE).exists() else None

def drill(sector, n=8, by="nd_equity"):
    if detail is None:
        print("companies_leverage.csv not found — rebuild with REFRESH_LEVERAGE = True."); return
    d = detail[detail["sector"] == sector].dropna(subset=[by]).sort_values(by, ascending=False)
    if d.empty:
        print(f"No companies with a valid {by} in {sector}."); return
    print(f"— {sector}: most-levered listed names (by {by}) —")
    print(d.head(n)[["name", "gd_assets", "nd_equity", "nd_ebitda", "assets"]].to_string(index=False))

if detail is not None:
    # Top 3 most-levered names per sector — the concrete drivers behind each bar.
    top = (detail.dropna(subset=["nd_equity"])
           .sort_values("nd_equity", ascending=False)
           .groupby("sector").head(3))
    print("Most-levered listed names by sector (top 3 each, by net debt / equity):\n")
    print(top[["sector", "name", "nd_equity", "gd_assets", "nd_ebitda"]].to_string(index=False))
    print("\nUse drill(\"<sector>\") to inspect one sector in full.")

## 9 · Leverage trend × NPL trend — is the sector deteriorating on both?

The most forward-looking cut. The x-axis is the SCR NPL 12-month trend (Phase 2); the
y-axis is the change in listed leverage between the two DFP vintages. **Top-right** is
where *both* credit signals are worsening — the sharpest deterioration read. Needs
`BUILD_TREND = True` (two DFP years).

In [ ]:
if panel is not None and "lev_trend" in panel and panel["lev_trend"].notna().any():
    t = panel.dropna(subset=["lev_trend", "npl_trend_12m_pp"])
    x, y = t["npl_trend_12m_pp"], t["lev_trend"]
    fig, ax = plt.subplots(figsize=(9.2, 6.6))
    ax.axhline(0, color="0.6", lw=1); ax.axvline(0, color="0.6", lw=1)
    ax.axhspan(0, y.max() + abs(y.max()) * 0.3 + 0.1, xmin=0.5, color=POS, alpha=0.07, zorder=0)
    ax.scatter(x, y, s=140, color=BAR, alpha=0.85, edgecolor="white", linewidth=1.2, zorder=3)
    placed = []
    for _, r in t.iterrows():
        px, py = r["npl_trend_12m_pp"], r["lev_trend"]
        crowded = any(abs(px - qx) < 0.06 and abs(py - qy) < 0.06 for qx, qy in placed)
        ax.annotate(r["sector"], (px, py), xytext=(0, -18 if crowded else 11),
                    textcoords="offset points", ha="center", va="top" if crowded else "bottom", fontsize=9)
        placed.append((px, py))
    ax.set_xlabel("SCR NPL trend — 12-month change (pp)  ·  worsening →")
    ax.set_ylabel(f"CVM leverage trend — Δ {LEV_METRIC}  ·  worsening ↑")
    ax.set_title("Both signals deteriorating? Loan book vs. listed balance sheets\n"
                 "top-right = credit turning on both lenses", fontsize=11)
    ax.margins(0.18); fig.tight_layout(); plt.show()
else:
    print("Leverage trend not available — set BUILD_TREND = True and REFRESH_LEVERAGE = True to compute it.")

## 10 · Reading & limitations

**How to read it.** The cross and map show *level* agreement; the composite score
ranks origination priority from both lenses; the divergence bar isolates where they
disagree (and why); the drill-down names the companies behind each sector; the trend
map flags where credit is turning on *both* the loan book and the listed balance
sheets at once.

**Limitations (honest).**
- **Universe mismatch — the big one.** SCR spans every firm with registered credit;
  CVM only B3-listed large-caps; the deals are mid-market. Three populations — read
  the cross as triangulation, not identity. The near-zero rank correlation is itself a
  reading: loan-book stress lives largely in firms the listed sample never sees.
- **EBITDA is approximated** (D&A by text on the indirect cash flow); `nd_ebitda` is
  indicative. `nd_equity` and `gd_assets` are clean from the balance sheet.
- **Financials distort leverage.** Banks are structurally leveraged and EBITDA is
  meaningless for them; treat "Serviços Financeiros" with care (or drop it).
- **Sector mapping is heuristic** on both sides; editable and explicit, but approximate.
- **The deal side stays synthetic.** This cross is real on both credit axes; tying it
  to deal *activity* still rests on the labeled synthetic deals.

Sources — CVM Dados Abertos (DFP): https://dados.cvm.gov.br/dataset/cia_aberta-doc-dfp ·
BCB SCR.data: https://dadosabertos.bcb.gov.br/dataset/scr_data